# 06 — RL Contextual Bandit for Prompt Template Selection

LinUCB contextual bandit learns which template yields the best reciprocal rank per query type. Compared against ε-greedy and random baselines.

**Requires API key.**

In [ ]:
import sys; sys.path.insert(0, '..')
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import Settings
from src.data.fb15k237 import FB15k237Dataset
from src.models.embedding_baseline import RotatEBaseline
from src.models.llm_client import LLMClient
from src.models.reranker import LLMReranker
from src.rl.bandit import LinUCBAgent, EpsilonGreedyAgent
from src.rl.features import extract_features
from src.rl.prompt_selector import PromptSelector
from src.eval.candidates import generate_candidates
from src.eval.metrics import compute_metrics
from src.wikidata.sparql import WikidataGrounder
from src.wikidata.cache import SPARQLCache
from src.utils.cost_tracker import CostTracker
from src.utils.reproducibility import set_seed

settings = Settings()
set_seed(settings.random_seed)
assert settings.ai_gateway_api_key, 'Set AI_GATEWAY_API_KEY in .env!'

## 1. Setup

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
embed_model = RotatEBaseline(cache_dir=settings.cache_dir)
embed_model.load()
cache    = SPARQLCache(cache_dir=settings.cache_dir)
grounder = WikidataGrounder(
    sparql_url=settings.wikidata_sparql_url,
    user_agent=settings.wikidata_user_agent,
    cache=cache,
)
client = LLMClient(settings=settings)
TEMPLATES = ['minimal','with_descriptions','strict_rubric','concise_cot','binary_judge']
n_arms    = len(TEMPLATES)
print(f'{n_arms} arms: {TEMPLATES}')

## 2. Verify Feature Extraction

In [ ]:
h,r,t = dataset.test[0]
feats  = extract_features((h,r,None), dataset.train)
print(f'Feature vector ({len(feats)}-dim): {feats}')

## 3. LinUCB Training Loop

In [ ]:
random.seed(settings.random_seed)
queries  = random.sample(dataset.test, settings.sample_test_queries)
linucb   = LinUCBAgent(n_arms=n_arms, feature_dim=8, alpha=1.0)
selector = PromptSelector(agent=linucb, templates=TEMPLATES)

history={'step':[],'reward':[],'arm':[],'cumulative':[]}
cumulative=0.0
for step,(h,r,t_true) in enumerate(queries):
    cands   = generate_candidates(model=embed_model,head=h,relation=r,
                                   dataset=dataset,top_k=settings.num_candidates)
    features= extract_features((h,r,None), dataset.train)
    template,arm_idx = selector.select(features)
    tt      = CostTracker()
    rr      = LLMReranker(client=client,grounder=grounder,cost_tracker=tt,template=template)
    reranked= rr.rerank(head=h,relation=r,candidates=[e for e,_ in cands])
    rank    = next((i+1 for i,(e,_) in enumerate(reranked) if e==t_true),len(reranked)+1)
    reward  = 1.0/rank - settings.rl_reward_lambda*tt.total_cost
    selector.update(arm_idx, features, reward)
    cumulative+=reward
    history['step'].append(step); history['reward'].append(reward)
    history['arm'].append(arm_idx); history['cumulative'].append(cumulative)
    print(f'  step {step+1:3d} | {template:<25} | rank={rank:3d} | reward={reward:.4f}')

## 4. Cumulative Reward & Template Frequency

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(history['step'],history['cumulative'],color='steelblue')
axes[0].set(title='Cumulative Reward (LinUCB)',xlabel='Step',ylabel='Cumulative reward')
arm_counts=pd.Series(history['arm']).value_counts().sort_index()
axes[1].bar([TEMPLATES[i] for i in arm_counts.index],arm_counts.values,color='coral')
axes[1].set(title='Template Selection Frequency',ylabel='Times selected')
plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()

## 5. LinUCB vs ε-Greedy vs Random

In [ ]:
def run_agent(agent, queries, contextual):
    rewards=[]
    for h,r,t_true in queries:
        cands=generate_candidates(model=embed_model,head=h,relation=r,
                                  dataset=dataset,top_k=settings.num_candidates)
        feats=extract_features((h,r,None),dataset.train)
        idx = agent.select(feats) if contextual else agent.select()
        tt  = CostTracker()
        rr  = LLMReranker(client=client,grounder=grounder,cost_tracker=tt,template=TEMPLATES[idx])
        rkd = rr.rerank(head=h,relation=r,candidates=[e for e,_ in cands])
        rnk = next((i+1 for i,(e,_) in enumerate(rkd) if e==t_true),len(rkd)+1)
        rew = 1.0/rnk - settings.rl_reward_lambda*tt.total_cost
        if contextual: agent.update(idx,feats,rew)
        else:          agent.update(idx,rew)
        rewards.append(rew)
    return rewards

random.seed(settings.random_seed)
linucb2  = LinUCBAgent(n_arms=n_arms,feature_dim=8,alpha=1.0)
egreedy2 = EpsilonGreedyAgent(n_arms=n_arms,epsilon=0.1)
rand_rew = [1.0/random.randint(1,20) for _ in queries]
lu_rew   = run_agent(linucb2,  queries, contextual=True)
eg_rew   = run_agent(egreedy2, queries, contextual=False)

fig,ax=plt.subplots(figsize=(9,4))
for rew,lbl,col in [(np.cumsum(lu_rew),'LinUCB','steelblue'),
                     (np.cumsum(eg_rew),'ε-Greedy','coral'),
                     (np.cumsum(rand_rew),'Random','gray')]:
    ax.plot(rew,label=lbl,color=col)
ax.set(title='Cumulative Reward Comparison',xlabel='Step',ylabel='Cumulative reward')
ax.legend(); plt.tight_layout(); plt.show()